# Targumim — Aramaic OT Translations

Verse-level Aramaic text for the major Targumim, downloaded from Sefaria.

**Prerequisites:** Run `python scripts/fetch_targum_data.py` once to populate `data/processed/targum.parquet`.

| Targum | Books Covered |
|---|---|
| Onkelos | Gen, Exo, Lev, Num, Deu |
| Jonathan | Jos, Jdg, Isa, Jer, Ezk, Hos, Amo, Mic, Nah, Hab, Zec |
| Psalms | Psa |

Note: Only verse-level text is available (no morphological tagging).
For word-level Peshitta morphology see `notebooks/nt/peshitta/peshitta_morphology.ipynb`.

In [ ]:
import sys
sys.path.insert(0, '../../../src')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from bible_grammar.targum_query import load_targum, COVERAGE

tg = load_targum()
print(f'Loaded {len(tg):,} verses')
print(tg.groupby(['targum','book_id']).size().to_string())

---
## 1. Verse Coverage by Book

In [ ]:
coverage = tg.groupby(['targum', 'book_id']).size().reset_index(name='verses')
print(coverage.to_string())

---
## 2. Onkelos — Torah Sample

In [ ]:
onkelos_gen1 = load_targum(targum='Onkelos', book_id='Gen')
onkelos_gen1 = onkelos_gen1[onkelos_gen1['chapter'] == 1].head(10)
for _, row in onkelos_gen1.iterrows():
    print(f"Onkelos Gen {row['chapter']}:{row['verse']}: {row['text']}")

---
## 3. Targum Jonathan — Isaiah Sample

In [ ]:
tj_isa = load_targum(targum='Jonathan', book_id='Isa')
ch53 = tj_isa[tj_isa['chapter'] == 53]
for _, row in ch53.iterrows():
    print(f"Tg. Isa {row['chapter']}:{row['verse']}: {row['text']}")

---
## 4. Targum to Psalms — Psalm 22

In [ ]:
tg_ps = load_targum(targum='Psalms', book_id='Psa')
ps22 = tg_ps[tg_ps['chapter'] == 22]
for _, row in ps22.iterrows():
    print(f"Tg. Ps {row['chapter']}:{row['verse']}: {row['text']}")

---
## 5. Keyword / Passage Search Across All Targumim

In [ ]:
# Search by substring in Aramaic text
QUERY = 'מֵימְרָא'  # Memra — the Word (divine speech hypostasis)

hits = tg[tg['text'].str.contains(QUERY, na=False)]
print(f'Found {len(hits)} verses containing "{QUERY}"')
print(hits.groupby(['targum', 'book_id']).size().to_string())
print()
for _, row in hits.head(10).iterrows():
    print(f"{row['targum']} {row['book_id']} {row['chapter']}:{row['verse']}: {row['text'][:80]}")

---
## 6. Cross-Reference — Compare MT, LXX, Targum, Peshitta for one verse

Combine all sources for a side-by-side comparison.

In [ ]:
import sys
sys.path.insert(0, '../../../src')
from bible_grammar.translations import load_translations
from bible_grammar.syntax_ot import load_syntax_ot

trans = load_translations()
ot = load_syntax_ot()

# Isaiah 53:5 — a messianic passage
BOOK, CH, VS = 'Isa', 53, 5

mt_words = ot[(ot['book'] == BOOK) & (ot['chapter'] == CH) & (ot['verse'] == VS)]
mt_text = ' '.join(mt_words['text'].tolist())

kjv_text = trans[(trans['translation']=='KJV') & (trans['book_id']==BOOK) & (trans['chapter']==CH) & (trans['verse']==VS)]['text'].values
kjv_text = kjv_text[0] if len(kjv_text) else ''

tg_text = tg[(tg['targum']=='Jonathan') & (tg['book_id']==BOOK) & (tg['chapter']==CH) & (tg['verse']==VS)]['text'].values
tg_text = tg_text[0] if len(tg_text) else 'N/A'

print(f'{BOOK} {CH}:{VS}')
print(f'MT:       {mt_text}')
print(f'KJV:      {kjv_text}')
print(f'Tg. Jon.: {tg_text}')